# Работа с данными бизнеса в ClickHouse

## Задание 1
```
SELECT usage_geo_id_name AS "Город",
       round(SUM(hours)) AS "Общее суммарное время",
       round(sumIf(hours, usage_platform_ru ILIKE '%IOS%')) AS "Cуммарное время IOS",
       round(sumIf(hours, usage_platform_ru ILIKE '%Android%')) AS "Cуммарное время Android"
FROM source_db.audition 
WHERE usage_geo_id_name NOT ILIKE '%Федеральный округ%' AND (usage_platform_ru ILIKE '%Букмейт IOS%' or usage_platform_ru ILIKE '%Букмейт Android%') AND usage_country_name='Россия'
GROUP BY usage_geo_id_name
ORDER BY "Общее суммарное время" DESC
LIMIT 20
```

По итогам запроса видим, что топ-1 городом по общему суммарному времени (26 629 часов) является Москва, при этом суммарное время на Android составило 18 573 часов, а на iOS 8 056 часов 

## Задание 2
```
SELECT main_content_name AS "Название книги",
       main_author_name AS "Автор",
       round(SUM(hours),2) AS "Суммарное время",
       round(avgIf(hours, main_content_type='Book'),2) AS "Среднее время чтения",
       round(avgIf(hours, main_content_type='Audiobook'),2) AS "Среднее время прослушивания"
FROM source_db.content c
JOIN source_db.audition a ON c.main_content_id=a.main_content_id 
WHERE main_content_type IN('Book','Audiobook') AND (usage_platform_ru ILIKE '%Букмейт IOS%' OR usage_platform_ru ILIKE '%Букмейт Android%')
GROUP BY "Название книги", "Автор"
HAVING uniqExact(main_content_type)=2
ORDER BY "Суммарное время" DESC
LIMIT 5
```

Самой популярной книгой в сервисе является "Илон Маск" за авторством Уолтера Айзексона. Суммарно ее читали и слушали 1 012.93 часа. Среднее время чтения за 1 сессию составило 0.29 часа, среднее время прослушивания - 0.69 часа.

## Задание 3
```
SELECT c.main_author_name AS "Автор",
       round(sum(a.hours), 2) AS "Общая длительность",
       uniqExactIf(c.main_content_id, c.main_content_type = 'Book') AS "Кол-во текстовых книг",
       round(avgIf(
       a.hours, 
       c.main_content_type = 'Audiobook' 
AND (a.usage_platform_ru ILIKE '%IOS%' OR a.usage_platform_ru ILIKE '%Android%')), 2) AS "Среднее время прослушивания   аудиокниг (Mobile)"
FROM source_db.content AS c
JOIN source_db.audition AS a ON c.main_content_id = a.main_content_id 
WHERE c.main_content_type IN ('Book', 'Audiobook') 
AND a.usage_platform_ru ILIKE '%Букмейт%'
GROUP BY "Автор"
HAVING countIf(c.main_content_type = 'Audiobook') > 0
ORDER BY "Общая длительность" DESC
LIMIT 10
```

Топ-1 автором по суммарной длительности чтения/прослушивания на всех платформах (3 201.95 часа) является Сергей Лукьяненко, количество текстовых книг за его авторством составило 54, а среднее время прослушивания аудиоверсий его книг на мобильных устройствах в среднем составило 1.76 часа


## Задание 4
```
WITH platform_stats AS (
SELECT puid,
       usage_platform_ru,
       sumIf(hours, main_content_type = 'Book') AS text_hours,
       sumIf(hours, main_content_type = 'Audiobook') AS audio_hours,
       sum(hours) AS platform_hours
FROM source_db.audition
INNER JOIN source_db.content USING (main_content_id)
WHERE usage_platform_ru ILIKE '%Букмейт Android%' OR usage_platform_ru ILIKE '%Букмейт iOS%'
GROUP BY puid, usage_platform_ru
),
user_aggregated AS (
SELECT puid,
       argMax(usage_platform_ru, platform_hours) AS raw_main_platform,
       sum(text_hours) AS text_hours,
       sum(audio_hours) AS audio_hours,
       text_hours + audio_hours AS content_hours
       FROM platform_stats
       GROUP BY puid
       HAVING content_hours > 0)
SELECT multiIf(
       raw_main_platform ILIKE '%Android%', 'Android',
       raw_main_platform ILIKE '%iOS%', 'iOS','Другое') AS main_platform,
       multiIf(
       audio_hours / content_hours >= 0.7, 'Слушатель',
       text_hours / content_hours >= 0.7, 'Читатель', 'Оба') AS segment,
       count() AS user_count
FROM user_aggregated
GROUP BY main_platform, segment
ORDER BY main_platform, user_count DESC
```

Приходим к выводу, что:
1. Количество слушателей (2 145) и читателей (2 382) на Android действительно примерно равное
2. На iOS читателей больше, чем слушателей (1 996 и 1 508 соответственно), но вдвое больше, как предполагал менеджер

## Задание 5
```
WITH daily_stats AS (
SELECT c.main_content_type AS content_type,
       a.msk_business_dt_str AS day,
       IF(DATENAME('weekday', day) IN ('Saturday', 'Sunday'), 'Выходные', 'Будни') AS weekday,
       SUM(hours) AS sum_hours
FROM source_db.audition a
JOIN source_db.content c ON a.main_content_id = c.main_content_id
WHERE c.main_content_type IN ('Audiobook','Book') AND usage_platform_ru ILIKE '%Букмейт%'
GROUP BY content_type, day, weekday)
SELECT DISTINCT content_type,
       round(avgIf(sum_hours, weekday = 'Выходные') OVER (PARTITION BY content_type)) AS avg_weekend_hours,
       round(avgIf(sum_hours, weekday = 'Будни') OVER (PARTITION BY content_type)) AS avg_work_hours
FROM daily_stats
ORDER BY content_type
```

Отвечая на вопрос: Падает ли использование аудиокниг в выходные на всех платформах, включая веб?, мы делаем вывод, что: да, падает, но незначительно, среднее время прослушивания составило 1 256 часов в выходные и 1 469 часов в рабочие дни.

## Задание 6
```
WITH user_versions AS (
SELECT usage_platform_ru AS platform,
       puid,
       argMax(app_version, msk_business_dt_str) AS user_latest_version 
FROM source_db.audition
WHERE usage_platform_ru ILIKE '%Букмейт iOS%' or usage_platform_ru ILIKE '%Букмейт Android%' 
GROUP BY platform, puid
),
platform_versions AS (
SELECT platform,
       MAX(user_latest_version) AS platform_max_version 
FROM user_versions
GROUP BY platform
)
SELECT u.platform,
       round(countIf(u.user_latest_version = p.platform_max_version) * 100.0 / count(u.puid), 2) AS latest_version_pct
FROM user_versions u
JOIN platform_versions p ON u.platform = p.platform
GROUP BY u.platform
ORDER BY u.platform
```

Итак, последняя версия приложения стоит у 28.92% пользователей Android и у 1.91% пользователей iOS

## Задание 7
```
SELECT platform,
       round(avg(updates_count), 2) AS update_rate
FROM (
SELECT usage_platform_ru AS platform,
       puid,
       uniqExact(app_version) - 1 AS updates_count
FROM source_db.audition
WHERE (usage_platform_ru ILIKE '%Букмейт Android%' OR usage_platform_ru ILIKE '%Букмейт iOS%') 
GROUP BY platform, puid
)
GROUP BY platform
ORDER BY platform
```

Средняя частота обновления среди пользователей Android составила 2.67, среди пользователей iOS - 2.09. Предположение о том, что пользователи iOS чаще обновляют приложение оказалось неверным

## Задание 8
```
SELECT countIf(has(published_topic_title_list, 'Магия')) as magic_books_count
FROM source_db.content
```

Всего книг с тегом "Магия" - 46 штук.

## Задание 9
```
SELECT countIf(main_content_name ILIKE '%магия%' 
AND NOT has(published_topic_title_list, 'Магия')
AND NOT has(published_topic_title_list, 'Художественная литература')
 ) as name_magic_no_tag
FROM source_db.content
```

Всего книг со словом "магия" в названии и без тега "Магия" и без учета тега "Художественная литература" - 49 штук.

## Задание 10
```
SELECT round(avg(length(published_topic_title_list)), 2) AS avg_categories_all,
       round(avgIf(length(published_topic_title_list), has(published_topic_title_list, 'Магия')), 2) AS avg_categories_magic
FROM source_db.content
```

Среднее количество категорий у книг с тегом "Магия" составило 3.22, среднее количество категорий у книг в целом составило 3.77. Получилось не превысить рекомендованное количество в виде 3-4 категорий на книгу.

## Задание 11
```
SELECT usage_country_name,
       usage_platform_ru,
       (stddevPop(hours_sessions_long) / avg(hours_sessions_long)) AS coef_variation
FROM source_db.audition
WHERE usage_platform_ru ILIKE 'Букмейт Android' OR usage_platform_ru ILIKE 'Букмейт iOS'
GROUP BY usage_country_name, usage_platform_ru 
HAVING avg(hours_sessions_long) > 0
ORDER BY coef_variation DESC
LIMIT 1
```

Самый большой коэффициент вариации составил 7.77 в стране Латвия в приложении на Android